In [21]:
import pandas as pd
import folium
import webbrowser
from datetime import datetime
import os
from folium.plugins import BoatMarker

from boats.lib.common import DIR_HOME
from boats.lib.boat import Boat
from boats.lib.common import MY_USER, get_boat_race_data, DIR_HTML

In [2]:
zoom_start = 8

In [3]:
myf_csv = os.path.join(DIR_HOME, 'boats', 'petsamo', 'MARK 3.csv')

In [4]:
myf_json = os.path.join(DIR_HOME, 'boats', 'petsamo', 'MARK 3.json')

In [5]:
with open(myf_csv, 'r') as f:
    contents = f.read()

with open(myf_csv, 'w') as f:
    f.write(contents.replace(';', ';,'))

df=pd.read_csv(myf_csv)

df.drop(0, axis=0, inplace=True)
df

In [7]:
df_json = pd.read_json(myf_json)
df_json.head()

,nom,tracks
0,MARK 3,"[1674766096, 132330.2850000000, -3858.7050000000]"
1,MARK 3,"[1674834600, 131377.3475534320, -3058.6823981734]"
2,MARK 3,"[1674861900, 131112.7549462747, -2499.2552013982]"
3,MARK 3,"[1674904200, 130853.2436410213, -1357.2159206937]"
4,MARK 3,"[1674914100, 130592.9875903700, -1252.0163772051]"


In [8]:
df_json.drop('nom', axis=1, inplace=True)
df_json.head()
df_json.drop(0, axis=0, inplace=True)

In [9]:
df_json['epoch'] = [df_json['tracks'][idx][0] for idx in df_json.index]
df_json['Lon'] = [float(df_json['tracks'][idx][1])/1000 for idx in df_json.index]
df_json['Lat'] = [float(df_json['tracks'][idx][2])/1000 for idx in df_json.index]
df_json.drop('tracks', axis=1, inplace=True)
df_json.head()

,epoch,Lon,Lat
1,1674834600,131.377348,-3.058682
2,1674861900,131.112755,-2.499255
3,1674904200,130.853244,-1.357216
4,1674914100,130.592988,-1.252016
5,1674927300,130.237354,-0.960540


In [10]:
df_json['epoch']=df_json['epoch'].astype(int)

In [11]:
df_json['ETA'] = [datetime.fromtimestamp(x).strftime("%d-%h %H:%M") for x in df_json['epoch']]
df_json.drop('epoch', axis=1, inplace=True)

In [12]:
df_json

,Lon,Lat,ETA
1,131.377348,-3.058682,27-Jan 10:50
2,131.112755,-2.499255,27-Jan 18:25
3,130.853244,-1.357216,28-Jan 06:10
4,130.592988,-1.252016,28-Jan 08:55
5,130.237354,-0.960540,28-Jan 12:35
6,130.088821,-0.704468,28-Jan 14:50
7,129.850827,0.238603,28-Jan 23:35
8,129.856286,0.621657,29-Jan 03:45
9,130.050904,2.729273,29-Jan 20:35
10,128.400326,8.872792,31-Jan 11:15


In [13]:
points = [(df_json.iloc[i]['Lat'], df_json.iloc[i]['Lon']) for i in range(len(df_json.index))]

In [14]:
mymap = folium.Map(location=[df_json.iloc[1]['Lat'], df_json.iloc[1]['Lon']], zoom_start=zoom_start)

In [19]:
boat_name = 'Petsamo'
race = 'Stardust'
user = 'Viper Vit'
oBoat = Boat(boat_name)
oBoat.getdata()
curr_pos = [round(oBoat.pos[0], 3), round(oBoat.pos[1], 3)]
sog = oBoat.nav['sog']
hdg = oBoat.nav['hdg']
race_data = get_boat_race_data(race, user, boat_name)
rank = race_data['rank']
track = race_data['track']
track.reverse()
track.append(curr_pos)

df_track = pd.DataFrame(track)
df_track.columns = ['Lat', 'Lon']

In [24]:
folium.PolyLine(points, color='red').add_to(mymap)
folium.PolyLine(track).add_to(mymap)
BoatMarker(curr_pos, color='blue',
           heading=oBoat.nav['hdg'],
           wind_heading=oBoat.wind['twd'],
           wind_speed=oBoat.wind['tws']).add_to(mymap)
mymap